# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [8]:
import langchain
print(langchain.__version__)

0.2.17


In [9]:
import warnings
warnings.filterwarnings('ignore')

In [10]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [11]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

In [16]:
!pip install pandas

   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.1/11.0 MB 1.1 MB/s eta 0:00:11
   -- ------------------------------------- 0.7/11.0 MB 6.5 MB/s eta 0:00:02
   -------------- ------------------------- 4.1/11.0 MB 25.9 MB/s eta 0:00:01
   --------------------- ------------------ 6.0/11.0 MB 34.6 MB/s eta 0:00:01
   ------------------------------ --------- 8.3/11.0 MB 35.3 MB/s eta 0:00:01
   ---------------------------------------  11.0/11.0 MB 65.6 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 54.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ------------ --------------------------- 3.9/12.6 MB 83.4 MB/s eta 0:00:01
   ------------------ --------------------- 5.7/12.6 MB 72.9 MB/s eta 0:00:01
   -------------------------- ------------- 8.5/12.6 MB 60.2 MB/s eta 0:00:01
   -------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.2.17 requires numpy<2,>=1; python_version < "3.12", but you have numpy 2.4.6 which is incompatible.
langchain-community 0.2.19 requires numpy<2,>=1; python_version < "3.12", but you have numpy 2.4.6 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [22]:
df.head(10)

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...
5,L'Or Espresso Café,Je trouve le goût médiocre. La mousse ne tient...
6,Hervidor de Agua Eléctrico,"Está lu bonita calienta muy rápido, es muy fun..."


## LLMChain

In [23]:
!pip install langchain_community

  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas 3.1.0 requires numpy>=2.0.2; python_version < "3.14", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [51]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [52]:
#Replace None by your own value and justify
llm = ChatOpenAI(temperature=0.5, model_name="gpt-4.1-mini")


- Chose temperature=0.5 to balance creativity and consistency

In [53]:
prompt = ChatPromptTemplate.from_template(template="Write a description for a {product}")
 #Write a query that would take a variable to describe any product

chain = prompt | llm

result = chain.invoke({"product": "L'Or Espresso Café"})
print(result) 


content="L'Or Espresso Café offers a rich and indulgent coffee experience, crafted from carefully selected Arabica and Robusta beans roasted to perfection. Known for its smooth, full-bodied flavor and aromatic intensity, L'Or Espresso delivers a luxurious taste that coffee lovers cherish. Whether enjoyed as a bold espresso shot or a creamy lungo, each cup promises a velvety texture and a lingering, satisfying finish. Ideal for those who appreciate premium quality and refined taste, L'Or Espresso Café transforms everyday coffee moments into a sophisticated ritual." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 16, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_4ffa17cc03', 'fin

In [54]:
chain = LLMChain(llm=llm, prompt=prompt)

In [55]:
product = "L'Or Espresso Café" # Select a product type to be described
chain.run(product)

"L'Or Espresso Café is a premium coffee experience that brings the rich, authentic taste of expertly crafted espresso to your daily routine. Known for its carefully selected Arabica and Robusta beans, L'Or Espresso delivers a full-bodied flavor with a smooth, velvety crema that enchants the senses. Whether enjoyed as a bold shot or a creamy latte, each cup promises a perfect balance of intensity and aroma, capturing the essence of traditional European coffee culture. Ideal for coffee lovers seeking quality and sophistication, L'Or Espresso Café transforms every moment into a luxurious coffee break."

## SimpleSequentialChain

In [57]:
from langchain.chains import SimpleSequentialChain

In [62]:
llm = ChatOpenAI(
    temperature=0.5,
    model_name="gpt-4.1-mini"
)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(template="Write a description for a {product}")

#Repeat the initial query or create a new query that would feed into the second prompt

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [63]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template("Based on the following product description, write marketing copy:\n\n{description}")

# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [64]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two], verbose=True)

In [65]:
overall_simple_chain.run(product)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


L'Or Espresso Café offers a refined coffee experience that celebrates the rich heritage of Italian espresso culture. Known for its premium quality beans sourced from the finest coffee-growing regions, L'Or Espresso delivers a perfectly balanced cup with intense aroma, deep flavor, and a smooth, velvety crema. Whether you prefer a bold ristretto, a classic espresso, or a creamy lungo, L'Or Espresso Café provides an inviting atmosphere where coffee enthusiasts can savor every sip. Ideal for those who appreciate craftsmanship and elegance in their daily coffee ritual.
Experience the art of Italian espresso with L'Or Espresso Café — where every cup is a celebration of rich heritage and premium quality. Sourced from the finest coffee-growing regions, our expertly crafted beans deliver an intense aroma, deep flavor, and a smooth, velvety crema that coffee lovers crave. Whether you choose a bold ristretto, a classic espresso, or a creamy lungo, L'Or Espresso Café invites you to indulge in a r

"Experience the art of Italian espresso with L'Or Espresso Café — where every cup is a celebration of rich heritage and premium quality. Sourced from the finest coffee-growing regions, our expertly crafted beans deliver an intense aroma, deep flavor, and a smooth, velvety crema that coffee lovers crave. Whether you choose a bold ristretto, a classic espresso, or a creamy lungo, L'Or Espresso Café invites you to indulge in a refined and elegant coffee ritual. Elevate your daily coffee moment with the perfect balance of craftsmanship and taste. Savor the essence of Italy in every sip."

**Repeat the above twice for different products**

In [66]:
prompt = ChatPromptTemplate.from_template(template="Write a description {product}")
 #Write a query that would take a variable to describe any product

chain = prompt | llm

result = chain.invoke({"product": "Luxury Air Mattress"})
print(result) 



content="Experience unparalleled comfort with our Luxury Air Mattress, designed to provide a plush and supportive sleeping surface wherever you go. Crafted from premium, durable materials, this mattress features a soft, velvety top layer that enhances breathability and ensures a cozy night's sleep. Its advanced air chamber technology offers customizable firmness, allowing you to easily adjust the mattress to your preferred level of support. Equipped with a built-in high-speed pump, inflation and deflation are quick and effortless. Perfect for guests, camping trips, or temporary bedding, the Luxury Air Mattress combines convenience with elegance, making every rest feel like a five-star experience." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 122, 'prompt_tokens': 13, 'total_tokens': 135, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_

In [67]:
chain = LLMChain(llm=llm, prompt=prompt)

In [68]:
product = "Luxury Air Mattress" # Select a product type to be described
chain.run(product)

"Experience unparalleled comfort and convenience with our Luxury Air Mattress, designed to provide a restful night's sleep wherever you go. Crafted with premium, durable materials, this air mattress features a plush, flocked top for a soft, cozy feel and enhanced grip for your sheets. Its advanced air retention technology ensures long-lasting firmness and support, while the built-in electric pump allows for effortless inflation and deflation in minutes. Ideal for guests, camping trips, or temporary sleeping arrangements, the Luxury Air Mattress combines portability with high-end comfort, making it the perfect choice for those who refuse to compromise on quality and ease."

In [69]:
llm = ChatOpenAI(temperature=0.5, model_name="gpt-4.1-mini")

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(template="Write a description for a {product}")

#Repeat the initial query or create a new query that would feed into the second prompt

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [70]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template("Based on the following product description, write marketing copy:\n\n{description}")

# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [71]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two], verbose=True)

In [72]:
overall_simple_chain.run(product)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Experience unparalleled comfort and convenience with our Luxury Air Mattress, designed to transform any space into a restful retreat. Crafted with premium, puncture-resistant materials and featuring a plush, flocked top surface, this air mattress offers exceptional support and softness for a restful night's sleep. Its built-in high-powered electric pump allows for effortless inflation and deflation in minutes, while adjustable firmness settings ensure personalized comfort tailored to your preference. Ideal for guests, camping, or temporary bedding, the mattress folds compactly for easy storage and transport. Elevate your sleeping experience with the perfect blend of durability, style, and luxury.
Transform any space into a serene sanctuary with our Luxury Air Mattress—the ultimate in comfort and convenience. Made from premium, puncture-resistant materials and topped with a plush, flocked surface, it delivers unbeatable support and softness for a truly restful night. Thanks to the built

'Transform any space into a serene sanctuary with our Luxury Air Mattress—the ultimate in comfort and convenience. Made from premium, puncture-resistant materials and topped with a plush, flocked surface, it delivers unbeatable support and softness for a truly restful night. Thanks to the built-in high-powered electric pump, inflation and deflation take just minutes, while adjustable firmness settings let you customize your perfect sleep. Whether hosting guests, camping, or needing temporary bedding, this mattress folds compactly for easy storage and transport. Elevate your sleep with durability, style, and luxury all in one. Experience comfort like never before!'

## SequentialChain

In [73]:
from langchain.chains import SequentialChain

In [81]:
llm = ChatOpenAI(temperature=0.5, model_name="gpt-4.1-mini")


first_prompt = ChatPromptTemplate.from_template("Translate the following review into Portuguese:\n\n{Review}")
  #This prompt should translate a review

chain_one = LLMChain(llm=llm, prompt=first_prompt, output_key="Portuguese_Review")


In [82]:
second_prompt = ChatPromptTemplate.from_template("Summarize the translated review:{Portuguese_Review}")
    #Write a promplt to summarize a review

chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="Summarize_Review") #give a name to this output


In [83]:
# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template("Translate the following review into English:{Review}")

# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="English_Review")


In [84]:
# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template("Write the review summary in both translated languages{Review}")
chain_four = LLMChain(llm=llm, prompt=fourth_prompt, output_key="Translated_Languages")


In [90]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(chains=[chain_one, chain_two, chain_three, chain_four], input_variables=["Review"], output_variables=["Portuguese_Review", "English_Review", "Summarize_Review"], verbose=False)

In [91]:
review = df.Review[5]
overall_chain(review)

{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\r\nVieux lot ou contrefaçon !?",
 'Portuguese_Review': 'Eu acho o sabor medíocre. A espuma não se mantém, é estranho. Compro os mesmos no comércio e o sabor é muito melhor...\nLote antigo ou falsificação!?',
 'English_Review': "I find the taste mediocre. The foam doesn't hold, it's strange. I buy the same ones in stores and the taste is much better...  \nOld batch or counterfeit!?",
 'Summarize_Review': "The reviewer finds the taste mediocre and notes that the foam doesn't hold, which feels unusual. They mention that the same product bought in stores tastes much better, suggesting this might be an old batch or a counterfeit."}

**Repeat the above twice for different products or reviews**

In [129]:
prompt = ChatPromptTemplate.from_template(template="Write a description {product}")
 #Write a query that would take a variable to describe any product

chain = prompt | llm

result = chain.invoke({"product": "Pillows Insert"})
print(result) 

content='A pillow insert is the inner cushion placed inside a decorative pillow cover, providing shape, support, and comfort. Typically filled with materials such as down feathers, memory foam, polyester fiberfill, or a blend of synthetic and natural fibers, pillow inserts come in various sizes and firmness levels to suit different preferences and uses. They are essential for maintaining the fullness and structure of throw pillows, bed pillows, and sofa cushions, enhancing both the aesthetic appeal and functionality of your home décor. Pillow inserts are often designed with removable covers for easy cleaning and come in standard dimensions to fit a wide range of pillowcases.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 13, 'total_tokens': 133, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_toke

In [130]:
chain = LLMChain(llm=llm, prompt=prompt)

In [131]:
product = "Pillows Insert" # Select a product type to be described
chain.run(product)

'A pillow insert is the inner cushion that goes inside a decorative pillow cover, providing shape, support, and comfort. Typically filled with materials such as polyester fiberfill, down feathers, memory foam, or a combination of these, pillow inserts vary in firmness and loft to suit different preferences. They come in various sizes and shapes to fit standard, euro, bolster, or specialty pillow covers. A high-quality pillow insert ensures that the outer pillow maintains its fullness and appearance over time, making it an essential component for both decorative and functional pillows.'

In [132]:
llm = ChatOpenAI(temperature=0.9, model_name="gpt-4.1-mini")

first_prompt = ChatPromptTemplate.from_template("Write a sarcastic comment about:\n\n{Review}")
  #This prompt should translate a review

chain_one = LLMChain(llm=llm, prompt=first_prompt, output_key="Sarcastic_Comment")

In [133]:
second_prompt = ChatPromptTemplate.from_template("Translate the review in Spanish:\n\n{Sarcastic_Comment}")
    #Write a promplt to summarize a review

chain_two = LLMChain(llm=llm, prompt=second_prompt, output_key="Spanish_Review") #give a name to this output

In [134]:
third_prompt = ChatPromptTemplate.from_template("Translate this review into Italian: {Review}")
chain_three = LLMChain(llm=llm, prompt=third_prompt, output_key="Italian_Review")

In [135]:
# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template("Write a final response using these two translations:\n\n"
    "Spanish: {Spanish_Review}\n\n"
    "Italian: {Italian_Review}")
chain_four = LLMChain(llm=llm, prompt=fourth_prompt, output_key="Translated_Languages")

In [136]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(chains=[chain_one, chain_two, chain_three, chain_four], input_variables=["Review"], output_variables=["Sarcastic_Comment", "Spanish_Review", "Italian_Review", "Translated_Languages"], verbose=False,)

In [137]:
review = df.Review[5]
overall_chain(review)

{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\r\nVieux lot ou contrefaçon !?",
 'Sarcastic_Comment': 'Ah oui, sûrement un vieux lot ou une contrefaçon sortie tout droit du grenier de grand-mère, parce que c’est bien connu que les fabricants adorent ruiner la qualité juste pour notre plaisir… incroyable, non ?!',
 'Spanish_Review': 'Ah sí, seguramente un lote viejo o una falsificación sacada directamente del desván de la abuela, porque es bien sabido que los fabricantes adoran arruinar la calidad solo para nuestro disfrute… increíble, ¿no?!',
 'Italian_Review': 'Trovo il gusto mediocre. La schiuma non regge, è strano. Compro gli stessi al supermercato e il gusto è molto migliore...  \nVecchio lotto o contraffazione!?',
 'Translated_Languages': 'Ah sí, seguramente un lote viejo o una falsificación sacada directamente del desván de la abuela, porque es bien sabido que los fabricantes adora

## Router Chain

In [138]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [139]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [140]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [141]:
llm = ChatOpenAI(temperature=0.5, model_name="gpt-4.1-mini")

In [142]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [143]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [144]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [145]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [149]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=False
                        )

In [150]:
chain.run("What is black body radiation?")

'Black body radiation is the electromagnetic radiation emitted by an idealized object called a black body, which absorbs all incident radiation regardless of frequency or angle. Because it absorbs all light, it also emits radiation with a characteristic spectrum that depends only on its temperature. This spectrum is continuous and shifts to shorter wavelengths as the temperature increases, described mathematically by Planck’s law. Black body radiation is fundamental in understanding thermal radiation and played a key role in the development of quantum mechanics.'

In [151]:
chain.run("what is 2 + 2")

"Let's break down the problem:\n\n1. Identify the numbers involved: 2 and 2.\n2. Understand the operation: addition (+).\n3. Add the two numbers together: 2 + 2.\n\nNow, performing the addition:\n\n2 + 2 = 4\n\nSo, the answer is **4**."

In [152]:
chain.run("Why does every cell in our body contain DNA?")

'Every cell in our body contains DNA because DNA holds the complete set of genetic instructions necessary for the development, functioning, and maintenance of that organism. Here’s a detailed explanation:\n\n1. **Genetic Blueprint:** DNA contains genes, which are sequences of nucleotides that encode the information to build proteins and RNA molecules. Proteins perform most of the structural, enzymatic, and regulatory functions in cells. Therefore, every cell needs access to this genetic blueprint to produce the proteins required for its specific functions.\n\n2. **Cell Specialization and Differentiation:** Although cells are specialized (e.g., muscle cells, nerve cells, skin cells), they all originate from a single fertilized egg and contain the same DNA. The difference in cell type arises from differential gene expression—different subsets of genes are turned on or off in each cell type. Having the full DNA allows each cell the potential to produce any protein if needed, depending on 

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [153]:
goalkeepers_template = """You are a goalkeepers coach. \
You are great at answering questions about goalkeeping skills\
and easy to understand. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


defenders_template = """You are a very good defenders coach. \
You are great at answering tactical questions. \
You are so good because you are experiencied and able to break down \
analysis about the oponent. 

Here is a question:
{input}"""

midfielders_template = """You are a very electric midfielders coach. \
You have an excellent knowledge of the human psycology.\
You have the ability to think, reflect, debate, discuss and \
evaluate the tactical dimension of the game. You have a respect and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


wingers_template = """ You are a successful retired winger.\
You have managed to play at a international level for 20 years.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of ofensive side of the game, and excellent communication \
skills.

Here is a question:
{input}"""

forwards_template = """You are an excellent forwards coach. \
You have a deep understanding of playing inside the box. \
You are skilled at observing patterns in players, analyzing scouting data, \
and explaining physiology, psycology of the player. \
You can clearly communicate well and you make connections between players.

Here is a question:
{input}"""

In [154]:
prompt_infos = [
    {
        "name": "goalkeeper", 
        "description": "Good for answering questions about goalkeepers", 
        "prompt_template": goalkeepers_template
    },
    {
        "name": "defenders", 
        "description": "Good for answering questions about the defense side", 
        "prompt_template": defenders_template
    },
    {
        "name": "midfielders", 
        "description": "Good for answering questions about the midfielders", 
        "prompt_template": midfielders_template
    },
    {
        "name": "wingers", 
        "description": "Good for answering wingers questions", 
        "prompt_template": wingers_template
    },
    {
        "name": "forwards",
        "description": "Good for answering questions about forwards",
        "prompt_template": forwards_template
    }
]

In [155]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [156]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [157]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [158]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [159]:
chain = MultiPromptChain(router_chain=router_chain, destination_chains=destination_chains, default_chain=default_chain, verbose=False)

In [160]:
chain.run("What is the most important tactical and ofensive actions for a midfielder in a 3-4-3 system")

'Thank you for the thoughtful question. As a coach specializing in electric midfielders, I appreciate the opportunity to delve deeply into the tactical and offensive responsibilities that define the role of a midfielder in a 3-4-3 system. This formation, known for its balance between width and central control, places unique demands on midfielders, who act as the engine room linking defense and attack.\n\n---\n\n### Tactical Context of the 3-4-3 System\n\nBefore detailing the midfielder’s offensive actions, it’s essential to understand the broader tactical framework:\n\n- **Shape:** The 3-4-3 typically features three central defenders, four midfielders (often two central midfielders and two wing-backs), and three forwards.\n- **Midfield Role:** The midfielders in the center are critical for controlling the tempo, transitioning the ball, and creating offensive opportunities while maintaining defensive solidity.\n- **Space Dynamics:** The system often relies on wing-backs to provide width

In [161]:
chain.run("What type of defensive mission should a striker have in a 3-4-3 system")

'Great question! The defensive mission of a striker in a 3-4-3 system is nuanced and crucial to the team’s overall defensive shape and transition phases. Let me break it down clearly, connecting tactical concepts, player psychology, and physiological demands.\n\n---\n\n### Understanding the 3-4-3 Defensive Structure\n\nIn a 3-4-3, the team typically relies on a back three, wing-backs, two central midfielders, and a front three. Defensively, this system often:\n\n- Compresses space centrally.\n- Uses wing-backs to cover wide areas.\n- Relies on forwards to initiate the first line of defense, pressing high and disrupting the opponent’s build-up.\n\n---\n\n### Defensive Mission of the Striker in 3-4-3\n\n**1. Lead the Pressing Line and Disrupt Build-Up**\n\n- **Primary role:** The striker (usually the central forward in the front three) acts as the spearhead of the high press.\n- **Objective:** Put immediate pressure on the opposition’s center-backs or deep-lying playmakers to force hurri

In [162]:
chain.run("What is the ideal central back profile for the 3-4-3 system")

'Great question! The 3-4-3 system relies heavily on the central defenders because they form the backbone of the defensive structure and initiate build-up play. The ideal central back profile in a 3-4-3 combines several key attributes and tactical awareness to ensure defensive solidity and fluid transition into attack.\n\n### Key Characteristics of the Ideal Central Back in a 3-4-3:\n\n#### 1. **Positional Intelligence and Communication**\n- In a back three, especially the central defender (often called the "sweeper" or "central stopper"), positional awareness is crucial.\n- They must constantly read the game, anticipate opponent movements, and organize the backline.\n- Communication with the wide center-backs and wing-backs is essential to maintain the defensive shape and cover spaces.\n\n#### 2. **Strong Aerial Presence**\n- The back three often face crosses and set-pieces where aerial duels are frequent.\n- The central CB should be dominant in the air to clear threats and win headers

In [163]:
chain.run("Who should play in the left and right side of the midfielders squad")

'Thank you for this thoughtful question. Selecting who should play on the left and right sides of the midfield involves a nuanced understanding of both the players’ individual attributes and the tactical framework of the team. It’s not simply about placing players on their preferred foot side but about maximizing the collective dynamic and exploiting the opposition’s weaknesses.\n\n### Key Considerations for Left and Right Midfield Roles\n\n1. **Player’s Natural Foot and Inverted Wingers:**  \n   Traditionally, a right-footed player on the right side and a left-footed player on the left side provide natural width and crossing options. However, modern tactics often employ *inverted wingers* — left-footed players on the right and right-footed players on the left — to cut inside and create goal-scoring opportunities or combine centrally with attacking midfielders or forwards.\n\n2. **Role within the Tactical System:**  \n   - **Wide Midfielder in a 4-4-2:** Usually tasked with providing w

In [164]:
chain.run("What type of recruitment process shoud we priorize when it comes to sign forwards to our squad")

'Great question! When prioritizing the recruitment process for forwards, especially those who will operate inside the box, it’s crucial to adopt a multi-dimensional approach that blends data analysis, scouting insights, physiological profiling, and psychological evaluation. Here’s a detailed breakdown of the recruitment priorities:\n\n### 1. **Define the Role and Playing Style First**\nBefore scouting or analyzing candidates, clarify the type of forward your team needs:\n- Are you looking for a traditional “poacher” who thrives on quick reactions and positioning inside the box?\n- A target man who can hold up play and bring others into the game?\n- A mobile, pressing forward who also drops deep to create space?\n\nThis clarity ensures recruitment is aligned with your tactical philosophy.\n\n---\n\n### 2. **Data-Driven Scouting: Key Metrics to Prioritize**\nFor inside-the-box forwards, quantitative data can highlight natural goal threats and movement patterns:\n- **Expected Goals (xG) a